# Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from _import import *

# Check for missing nodes (raw data)

In [ ]:
# For raw df:

# Check which nodes have no data for the raw stress and deformation variables
df_stress = read_data_file(0, 0, 0, 0, 4, filter_out_invalid_nodes=False)
df_deformation = read_data_file(0, 0, 0, 0, 0, filter_out_invalid_nodes=False)
# Get nodes missing loads for stress and deformation
def check_missing_loads(df_loads, variable_name):
    missing_loads_nodes = df_loads[df_loads.isnull().any(axis=1)]["Node Number"].unique()
    if len(missing_loads_nodes) > 0:
        print(f"Warning: {len(missing_loads_nodes)} nodes are missing loads for {variable_name}.")
    return missing_loads_nodes
missing_loads_deformation = check_missing_loads(df_deformation, "deformation")
missing_loads_stress = check_missing_loads(df_stress, "stress")
# Get node locations of missing stress
missing_stress_nodes = COORDS_DF[COORDS_DF["Node Number"].isin(missing_loads_stress)]["Node Number"].unique()
missing_stress_nodes


Save to file

In [ ]:

# Save nodes without stress
with open("nodes_missing_stress.txt", "w") as f:
    for node in sorted(missing_stress_nodes):
        f.write(f"{node}\n")
# Save nodes without deformation (and no coordinates)
with open("nodes_missing_deformation.txt", "w") as f:
    for node in sorted(missing_loads_deformation):
        f.write(f"{node}\n")
# Node numbers to be used have both coordinates and loads
valid_node_numbers = set(NODE_NUMBERS) - set(missing_loads_deformation)
assert len(valid_node_numbers) == len(NODE_NUMBERS) - len(missing_loads_deformation)
print(f"{len(valid_node_numbers)} Node numbers have both coordinates and loads.")
with open("valid_node_numbers.txt", "w") as f:
    for node in sorted(valid_node_numbers):
        f.write(f"{node}\n")

# Check for missing nodes (variable aggregated df)

In [ ]:
df = get_data_variable_aggregated(
    (0, 0, 0, 0), filter_out_invalid_nodes=False
)

In [ ]:
# For aggregated df:

# Check which node numbers have missing coordinates
coords = df["X"]
missing_coords_nodes = df[coords.isna()]["Node Number"].unique()

# Check that no coords nodes are the same as no deformation nodes
assert set(missing_coords_nodes) == set(missing_loads_deformation), "Mismatch between missing coords and deformation loads nodes"

# Check which nodes have missing loads
loads = df[VARIABLE_NAMES]
missing_loads_nodes = df[loads.isna().any(axis=1)]["Node Number"].unique()

print(f"{len(missing_coords_nodes)} Node numbers with missing coordinates:", missing_coords_nodes)
print(f"{len(missing_loads_nodes)} Node numbers with missing loads:", missing_loads_nodes)

loads_no_coords = set(missing_loads_nodes) - set(missing_coords_nodes)
coords_no_loads = set(missing_coords_nodes) - set(missing_loads_nodes)
print(f"{len(loads_no_coords)} Node numbers with missing loads but have coordinates:", loads_no_coords)
print(f"{len(coords_no_loads)} Node numbers with missing coordinates but have loads:", coords_no_loads)

In [ ]:
# Check which sequential node numbers are missing
all_node_numbers = set(range(df["Node Number"].min(), df["Node Number"].max() + 1))
present_node_numbers = set(df["Node Number"].unique())
missing_node_numbers = all_node_numbers - present_node_numbers
print("Missing node numbers:", missing_node_numbers)
# -> None are missing!